In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# AI transpiler passes

AI transpiler passes คือ passes ที่ทำหน้าที่แทนที่ passes แบบ "ดั้งเดิม" ของ Qiskit สำหรับงาน transpiling บางอย่าง โดยมักให้ผลลัพธ์ที่ดีกว่าอัลกอริทึมฮิวริสติกที่มีอยู่ (เช่น depth ต่ำกว่าและจำนวน CNOT น้อยกว่า) และยังเร็วกว่าอัลกอริทึมการปรับแต่งอย่าง Boolean satisfiability solvers มาก AI transpiler passes สามารถรันได้ในสภาพแวดล้อมเครื่องของตัวเองหรือบนคลาวด์ผ่าน Qiskit Transpiler Service หากเป็นส่วนหนึ่งของ IBM Quantum&reg; Premium Plan, Flex Plan หรือ On-Prem (via IBM Quantum Platform API) Plan

> **Note:** AI transpiler passes อยู่ในสถานะ beta release ซึ่งอาจมีการเปลี่ยนแปลง
>     หากมีข้อเสนอแนะหรือต้องการติดต่อทีมพัฒนา สามารถใช้ [Qiskit Slack Workspace channel](https://qiskit.slack.com/archives/C06KF8YHUAU) นี้ได้เลย

passes ที่ใช้งานได้ในปัจจุบันมีดังนี้:

**Routing passes**
 - `AIRouting`: การเลือก layout และการ routing ของ Circuit

**Circuit synthesis passes**
 - `AICliffordSynthesis`: การ synthesis ของ Clifford Circuit
 - `AILinearFunctionSynthesis`: การ synthesis ของ Linear Function Circuit
 - `AIPermutationSynthesis`: การ synthesis ของ Permutation Circuit
 - `AIPauliNetworkSynthesis`: การ synthesis ของ Pauli Network Circuit

ในการใช้งาน AI transpiler passes ต้อง[ติดตั้ง package `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service) ก่อน และสามารถดูข้อมูลเพิ่มเติมเกี่ยวกับตัวเลือกต่าง ๆ ได้ที่ [qiskit-ibm-transpiler API documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)

## รัน AI transpiler passes บนเครื่องหรือบนคลาวด์
ถ้าต้องการใช้ AI transpiler passes ในสภาพแวดล้อมเครื่องของตัวเองแบบฟรี ให้ติดตั้ง `qiskit-ibm-transpiler` พร้อม dependencies เพิ่มเติมดังนี้:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

หากไม่มี dependencies เพิ่มเติมเหล่านี้ AI transpiler passes จะรันบนคลาวด์ผ่าน Qiskit Transpiler Service (ใช้งานได้เฉพาะผู้ใช้ IBM Quantum Premium Plan, Flex Plan หรือ On-Prem (via IBM Quantum Platform API) Plan เท่านั้น) เมื่อติดตั้ง dependencies เพิ่มเติมแล้ว โหมดเริ่มต้นในการรัน AI transpiler passes จะใช้เครื่องของตัวเอง

## AI routing pass
`AIRouting` pass ทำหน้าที่ทั้งเป็น layout stage และ routing stage สามารถใช้ภายใน `PassManager` ได้ดังนี้:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

ในที่นี้ `backend` กำหนด coupling map ที่จะใช้ routing, `optimization_level` (1, 2 หรือ 3) กำหนดความพยายามในการคำนวณที่จะใช้ในกระบวนการ (ค่าสูงกว่ามักให้ผลดีกว่าแต่ใช้เวลานานกว่า) และ `layout_mode` กำหนดวิธีจัดการการเลือก layout
`layout_mode` มีตัวเลือกดังนี้:

- `keep`: รักษา layout ที่กำหนดโดย transpiler passes ก่อนหน้า (หรือใช้ trivial layout หากไม่ได้กำหนด) มักใช้เฉพาะเมื่อ Circuit ต้องรันบน qubits เฉพาะของอุปกรณ์ มักให้ผลลัพธ์ที่แย่กว่าเพราะมีพื้นที่สำหรับการปรับแต่งน้อยกว่า
- `improve`: ใช้ layout ที่กำหนดโดย transpiler passes ก่อนหน้าเป็นจุดเริ่มต้น เหมาะเมื่อมีการเดา layout เริ่มต้นที่ดี เช่น Circuit ที่สร้างขึ้นในลักษณะที่ใกล้เคียงกับ coupling map ของอุปกรณ์ ยังเหมาะหากต้องการลอง layout passes อื่น ๆ ร่วมกับ `AIRouting` pass
- `optimize`: นี่คือโหมดเริ่มต้น ทำงานได้ดีที่สุดสำหรับ Circuit ทั่วไปที่อาจไม่มีการเดา layout ที่ดี โหมดนี้จะละเว้นการเลือก layout ก่อนหน้า
- `local_mode`: flag นี้กำหนดว่า `AIRouting` pass จะรันที่ไหน ถ้าเป็น `False`, `AIRouting` จะรันจากระยะไกลผ่าน Qiskit Transpiler Service ถ้าเป็น `True` package จะพยายามรัน pass ในสภาพแวดล้อมเครื่องของตัวเอง โดยถ้าไม่พบ dependencies ที่จำเป็นจะ fallback ไปใช้โหมดคลาวด์

## AI circuit synthesis passes
AI circuit synthesis passes ช่วยให้ปรับแต่งส่วนต่าง ๆ ของ Circuit ประเภทต่าง ๆ ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) ด้วยการ re-synthesize วิธีทั่วไปในการใช้ synthesis pass มีดังนี้:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

การ synthesis จะเคารพ coupling map ของอุปกรณ์ โดยสามารถรันได้อย่างปลอดภัยหลัง routing passes อื่น ๆ โดยไม่รบกวน Circuit ดังนั้น Circuit โดยรวมจะยังคงเป็นไปตามข้อจำกัดของอุปกรณ์ ตามค่าเริ่มต้น การ synthesis จะแทนที่ sub-circuit เดิมเฉพาะเมื่อ sub-circuit ที่ synthesize ได้ดีกว่าเดิม (ปัจจุบันตรวจสอบเฉพาะจำนวน CNOT) แต่สามารถบังคับให้แทนที่เสมอได้โดยตั้งค่า `replace_only_if_better=False`

synthesis passes ต่อไปนี้สามารถ import ได้จาก `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis สำหรับ [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) circuits (บล็อกของ Gate `H`, `S` และ `CX`) ปัจจุบันรองรับสูงสุดเก้า qubit blocks
- *AILinearFunctionSynthesis*: Synthesis สำหรับ [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (บล็อกของ Gate `CX` และ `SWAP`) ปัจจุบันรองรับสูงสุดเก้า qubit blocks
- *AIPermutationSynthesis*: Synthesis สำหรับ [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (บล็อกของ Gate `SWAP`) ปัจจุบันรองรับ 65, 33 และ 27 qubit blocks
- *AIPauliNetworkSynthesis*: Synthesis สำหรับ Pauli Network circuits (บล็อกของ Gate `H`, `S`, `SX`, `CX`, `RX`, `RY` และ `RZ`) ปัจจุบันรองรับสูงสุดหก qubit blocks

เราคาดว่าจะค่อย ๆ เพิ่มขนาดของ blocks ที่รองรับขึ้นเรื่อย ๆ

ทุก passes ใช้ thread pool เพื่อส่งคำขอหลายรายการพร้อมกัน ตามค่าเริ่มต้น จำนวน threads สูงสุดคือจำนวน cores บวกสี่ (ค่าเริ่มต้นของ Python object `ThreadPoolExecutor`) อย่างไรก็ตาม สามารถกำหนดค่าเองได้ด้วย argument `max_threads` เมื่อสร้าง pass ตัวอย่างเช่น บรรทัดต่อไปนี้สร้าง `AILinearFunctionSynthesis` pass ที่ใช้ threads สูงสุด 20 threads